<a href="https://colab.research.google.com/github/MuwafagQ/Playbook-program/blob/claude%2Fsetup-gpu-video-testing-JhgUH/colab_gpu_test%20(3.3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playbook Soccer Analytics — GPU Test (Google Colab)

**Before running:** Go to `Runtime → Change runtime type` and select **T4 GPU**.

You will need:
- A **Roboflow API key** (free at roboflow.com) stored in Colab Secrets as `ROBOFLOW_API_KEY`
- A short soccer video clip (MP4, ideally 10–30 seconds for a quick test)

**Pipeline highlights (good-baseline-may9):**
- BoTSort tracker + appearance ReID (`yolo11n-cls.pt`)
- IDStabilizer — re-links IDs after occlusions using position + torso appearance
- Color-based team classification (fast, no GPU needed for this step)
- BallSmoother — interpolates missing ball detections
- HomographyStateMachine — holds last good homography through short failures
- KPI summary JSON/CSV alongside the annotated video

In [1]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print('GPU detected:', gpu.stdout.strip())
else:
    print('⚠️  No GPU found.\n'
          'Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')

GPU detected: Tesla T4, 15360 MiB


In [2]:
# ── Cell 2: System packages ───────────────────────────────────────────────────
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev

In [3]:
# ── Cell 3: Python dependencies ───────────────────────────────────────────────
# Enable GPU first: Runtime ▸ Change runtime type ▸ T4 GPU

# Remove any existing onnxruntime installs to prevent CPU/GPU conflicts
!pip uninstall -y onnxruntime onnxruntime-gpu -q 2>/dev/null || true

# Swap out Colab's opencv for headless (avoids display-backend conflicts)
!pip uninstall -qqy opencv-python opencv-python-headless 2>/dev/null

!pip install -q \
    'numpy>=2.0.0,<2.4.0' \
    opencv-python-headless==4.10.0.84 \
    onnxruntime-gpu==1.20.1 \
    tqdm \
    'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' \
    pydantic-settings==2.4.0 \
    python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' \
    'inference==1.2.2' \
    'ultralytics>=8.4.37,<8.5.0' \
    'lap>=0.5.13,<0.6'

!pip install -q 'transformers>=5.2.0,<5.3.0'

# Roboflow sports library (color-team helper, pitch config, annotators)
!pip install -q git+https://github.com/roboflow/sports.git@main

print('\n✅ All packages installed.')


In [4]:
!pip install -q pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.2 MB/s eta 0:00:00


In [5]:
# ── Cell 3b: Assert CUDAExecutionProvider is available ────────────────────────
import onnxruntime as ort

providers = ort.get_available_providers()
print('ONNX providers:', providers)

assert 'CUDAExecutionProvider' in providers, (
    "\n\n❌ CUDAExecutionProvider not available!\n"
    "   Fix: Runtime ▸ Change runtime type ▸ T4 GPU, then Run All.\n"
    f"   Available providers: {providers}"
)

print('\n✅ ONNX GPU ready — CUDAExecutionProvider confirmed.')


In [7]:
# ── Cell 4: Clone the repo ────────────────────────────────────────────────────
# Public repo — no token needed. If private, replace with:
#   !git clone https://<YOUR_TOKEN>@github.com/muwafagq/playbook-program.git /content/playbook
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
!git clone --branch {BRANCH} https://github.com/muwafagq/playbook-program.git /content/playbook
!git pull origin claude/setup-gpu-video-testing-JhgUH
import os, sys
os.chdir('/content/playbook')
sys.path.insert(0, '/content/playbook')
active_branch = !git rev-parse --abbrev-ref HEAD
print('Working dir:', os.getcwd())
print('Branch:', active_branch[0])

Cloning into '/content/playbook'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 116 (delta 47), reused 53 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 5.13 MiB | 9.10 MiB/s, done.
Resolving deltas: 100% (47/47), done.
fatal: not a git repository (or any of the parent directories): .git
Working dir: /content/playbook
Branch: claude/setup-gpu-video-testing-JhgUH


In [23]:
!git pull origin claude/setup-gpu-video-testing-JhgUH

From https://github.com/muwafagq/playbook-program
 * branch            claude/setup-gpu-video-testing-JhgUH -> FETCH_HEAD
Already up to date.


In [8]:
import shutil, os, sys, torch
shutil.copy('baseline.env', '.env')

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    raise RuntimeError(
        "ROBOFLOW_API_KEY not found in Colab Secrets.\n"
        "Add it via the key icon in the left sidebar."
    )

# Patch the API key into .env
with open('.env', 'r') as f:
    env_text = f.read()
env_text = env_text.replace('ROBOFLOW_API_KEY=', f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}')
with open('.env', 'w') as f:
    f.write(env_text)

# Export environment variables
from dotenv import dotenv_values
env_vals = dotenv_values('.env')
for k, v in env_vals.items():
    if v is not None: os.environ[k] = v

# FORCE GPU CONFIGURATION
os.environ['DEVICE'] = 'cuda'
os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY

# Attempt to force Roboflow Inference to use CUDA provider
try:
    import inference.core.devices.utils as dev_utils
    import onnxruntime as ort
    dev_utils.GLOBAL_DEVICE = 'cuda'
    # Check if CUDA is actually available to ONNX
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        print("⚠️ ONNX doesn't see CUDA. Speed might be limited.")
except:
    pass

print('\n── Active model config ─────────────────────────────────')
print(f"CUDA Available (PyTorch): {torch.cuda.is_available()}")
print('DEVICE          :', os.environ.get('DEVICE'))
print('────────────────────────────────────────────────────────')


[06/08/26 14:55:59] WARNING  Your inference package version 1.2.2 is out of date! Please upgrade to  ]8;id=10024694;file:///usr/local/lib/python3.12/dist-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=10024695;file:///usr/local/lib/python3.12/dist-packages/inference/core/__init__.py#41\41]8;;\
                             version 1.3.0 of inference for the latest features and bug fixes by                   
                             running `pip install --upgrade inference`.                                            


── Active model config ─────────────────────────────────
CUDA Available (PyTorch): True
DEVICE          : cuda
────────────────────────────────────────────────────────


In [9]:
import torch, gc, sys

# 1. Clear GPU memory and cache
gc.collect()
torch.cuda.empty_cache()

# 2. Force reload the main module and its vision components to pick up CPU changes
import importlib
modules_to_reload = ['main', 'vision.detect', 'vision.utils']

for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

print('✅ Modules reloaded. Ready to run on CPU.')

✅ Modules reloaded. Ready to run on CPU.


In [10]:
# @title
# ── Cell 6: Provide a test video ──────────────────────────────────────────────
# Choose ONE option and comment out the others.

# --- Option A: Upload a local file -------------------------------------------
from google.colab import files as colab_files
print('Select your MP4 file in the dialog below...')
uploaded = colab_files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]
print('Video ready at:', VIDEO_PATH)

# --- Option B: Download a YouTube clip (yt-dlp) ------------------------------
# !pip install -q yt-dlp
# YT_URL = 'https://www.youtube.com/watch?v=REPLACE_ME'
# !yt-dlp -o /content/test_clip.%(ext)s --recode-video mp4 -q "$YT_URL"
# VIDEO_PATH = '/content/test_clip.mp4'

# --- Option C: Mount Google Drive --------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_PATH = '/content/drive/MyDrive/YOUR_FOLDER/your_clip.mp4'

Select your MP4 file in the dialog below...


Saving HILAL-HAZM_match_B_up7 - Trim.mp4 to HILAL-HAZM_match_B_up7 - Trim.mp4
Video ready at: /content/HILAL-HAZM_match_B_up7 - Trim.mp4


In [ ]:
# ── Cell 7 (optional): Trim to first N seconds ────────────────────────────────
# Skip if your clip is already short (< 30 s).
#TRIM_SECONDS = 20
#TRIMMED_PATH = '/content/playbook/test_trimmed.mp4'
#!ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c copy "{TRIMMED_PATH}" -loglevel warning
#VIDEO_PATH = TRIMMED_PATH
#print(f'Trimmed to {TRIM_SECONDS}s → {VIDEO_PATH}')

In [28]:
# ── Cell 8: Run the pipeline (GPU Verification) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, torch, gc, importlib

# Fix Pillow version conflict while maintaining NumPy 2.x compatibility
!pip install --force-reinstall Pillow==11.1.0 "numpy>=2.0.0,<2.1.0"
importlib.invalidate_caches()

import main

# Ensure environment is strictly set to CUDA
os.environ['DEVICE'] = 'cuda'

# Hardware check before running
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected by PyTorch. Please check Runtime type.")

gc.collect()
torch.cuda.empty_cache()

importlib.reload(main)

OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
VIDEO_INPUT = '/content/HILAL-HAZM_match_B_up7.mp4'

print(f"🚀 Starting GPU Pipeline: {VIDEO_INPUT}")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

try:
    main.main(
        source_video=VIDEO_INPUT,
        out_dir=OUT_DIR,
        enable_team=True,
    )
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")

  Using cached pillow-11.1.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached pillow-11.1.0-cp312-cp312-manylinux_2_28_x86_64.whl (4.5 MB)
Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.2 MB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.1.0
    Uninstalling pillow-11.1.0:
      Successfully uninstalled pillow-11.1.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inference 1.2.2 requires onnxruntime<1.22.0,>=1.15.1, which is not installed.
inference 1.2.2 requires filelock<=3.17.0,>=3.12.0, but you have filelock 3.29.0 which i

🚀 Starting GPU Pipeline: /content/HILAL-HAZM_match_B_up7.mp4
Using GPU: Tesla T4
[stage] Loading models...
[stage] Models loaded.
[stage] Opening video: /content/HILAL-HAZM_match_B_up7.mp4
[stage] Video ready. total_frames=862, processing=862
WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.
[stage] Tracker mode: botsort
[stage] Color team classifier enabled (init_samples=30, margin=0.08).
[stage] Starting frame loop...


  0%|          | 1/862 [00:00<10:15,  1.40it/s]

[homography] reject reason=inlier_ratio_low (n=12 ratio=0.33 min=0.50)


  0%|          | 2/862 [00:01<10:12,  1.40it/s]

...

[homography] reject reason=too_few_kp (n=4 min_kp=6)


 51%|█████▏    | 442/862 [05:33<05:43,  1.22it/s]

[homography] reject reason=too_few_kp (n=4 min_kp=6)


100%|██████████| 862/862 [10:52<00:00,  1.32it/s]

Done. Output video: /content/outputs/annotated.mp4
CSV: /content/outputs/per_frame_tracks.csv
Processed frames: 862
Avg tracked players/frame: 13.42
Ball detected frames: 650/862
Ball interpolated frames: 206/862
Homography OK frames: 213/862
Homography available frames: 657/862
[homography] failure summary: {'inlier_ratio_low': 379, 'ill_conditioned': 92, 'h_discontinuity': 130, 'too_few_kp': 40}
Valid projection rows: 9056/13640
Elapsed: 652.91s | Effective FPS: 1.32
KPI JSON: /content/outputs/kpi_summary.json
KPI CSV: /content/outputs/kpi_summary.csv


In [29]:
# ── Cell 10: KPI summary ──────────────────────────────────────────────────────
import json, pandas as pd

kpi_json = OUT_DIR + '/kpi_summary.json'
if os.path.exists(kpi_json):
    with open(kpi_json) as f:
        kpi = json.load(f)
    print(json.dumps(kpi, indent=2))
else:
    print('kpi_summary.json not found — check OUT_DIR path')

csv_path = OUT_DIR + '/per_frame_tracks.csv'
df = pd.read_csv(csv_path)
print(f'\nTracking CSV: {len(df):,} rows | {df.frame.nunique()} frames | {df.track_id.nunique()} unique IDs')
df.head(5)

{
  "source_video": "/content/HILAL-HAZM_match_B_up7.mp4",
  "processed_frames": 862,
  "effective_fps": 1.3202375886604834,
  "avg_players_per_frame": 13.417633410672854,
  "ball_detect_coverage": 0.7540603248259861,
  "ball_interp_coverage": 0.23897911832946636,
  "homography_ok_rate": 0.2470997679814385,
  "homography_available_rate": 0.7621809744779582,
  "valid_projection_ratio": 0.6639296187683285,
  "unique_tracks": 31,
  "short_track_ratio": 0.0,
  "team_unknown_ratio": 0.0
}

Tracking CSV: 13,640 rows | 862 frames | 32 unique IDs


,frame,track_id,display_track_id,class_id,conf,x1,y1,x2,y2,x_m,y_m,team_id,homography_ok,kp_used,inlier_ratio,reproj_err,detector_ran,homography_state,ball_interpolated
0,0,1,1,2,0.945949,78.0,785.0,107.0,890.0,NaN,NaN,-1,False,12,0.333333,1.000000e+09,1,none,0
1,0,2,2,2,0.922429,0.0,667.0,29.0,756.0,NaN,NaN,-1,False,12,0.333333,1.000000e+09,1,none,0
2,0,3,3,2,0.914495,1475.0,606.0,1500.0,687.0,NaN,NaN,-1,False,12,0.333333,1.000000e+09,1,none,0
3,0,4,4,2,0.913524,1745.0,536.0,1768.0,603.0,NaN,NaN,-1,False,12,0.333333,1.000000e+09,1,none,0
4,0,5,5,2,0.911785,151.0,512.0,175.0,576.0,NaN,NaN,-1,False,12,0.333333,1.000000e+09,1,none,0


In [30]:
# ── Cell 11: Download all outputs ─────────────────────────────────────────────
from google.colab import files as colab_files
for fname in ['annotated.mp4', 'per_frame_tracks.csv', 'kpi_summary.json', 'kpi_summary.csv']:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        colab_files.download(fpath)
    else:
        print(f'Skipped (not found): {fpath}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>